# 7주차 — 폐루프 완성 + ★브리지: 내 엔진 ↔ 진짜 blocksim★

**완성된 엔진 (1~6주차의 전부)**
```
ref → [err] → [PI 제어기] → [플랜트 (적분기 2개)] → scope
  ▲──────────────── 피드백 ─────────────────────────┘
```
mybs.py = 지난 6주의 완성본 (제공). 오늘은 이 엔진으로 루프를 닫고,
같은 모델을 정식 blocksim 으로 돌려 **6자리까지 같은 숫자**를 확인한다.
그 순간이 전반부의 졸업식이다.

## 이론 요약
- P 만: 빠르지만 정상상태 오차가 남는다 (오차가 있어야 출력이 나오므로)
- I: 오차를 누적 — 오차가 0 이 될 때까지 밀어붙인다
- 플랜트 1/(s²+2s+1) 를 미분방정식 ẍ = u - 2ẋ - x 로 풀어 적분기 2개로 짓는다
  (전달함수 블록 없이도 같은 시스템 — 상태공간의 감각)

In [ ]:
import importlib, mymodel; importlib.reload(mymodel)
import mybs, matplotlib.pyplot as plt
bd, sc = mymodel.build_pi_loop()
mybs.simulate(bd, 10.0, 0.01)
plt.plot(sc.t, sc.data[0]); plt.grid(True)
print('내 엔진  y(10s) =', f'{sc.data[0][-1]:.6f}',  ' (기준값 1.000189)')

In [ ]:
# ★브리지: 같은 문제를 정식 blocksim(전달함수 플랜트)으로
import sys; sys.path.insert(0, '../..')
import blocksim as bs
bd2 = bs.Diagram()
ref = bd2.add(bs.Step(t_on=1.0)); err = bd2.add(bs.Sum('+-'))
kp = bd2.add(bs.Gain(3.0)); it = bd2.add(bs.Integrator()); ki = bd2.add(bs.Gain(2.0))
pi = bd2.add(bs.Sum('++')); plant = bd2.add(bs.TransferFunction([1],[1,2,1]))
sc2 = bd2.add(bs.Scope(n_in=1))
bd2.connect(ref, err[0]); bd2.connect(plant, err[1]); bd2.connect(err, kp)
bd2.connect(err, it); bd2.connect(it, ki); bd2.connect(kp, pi[0]); bd2.connect(ki, pi[1])
bd2.connect(pi, plant); bd2.connect(plant, sc2)
bs.run(bd2, 10.0, 0.01)
print('blocksim y(10s) =', f'{sc2.data[0][-1]:.6f}')
print('최대차 =', max(abs(a-b) for a,b in zip(sc.data[0], sc2.data[0])))

## 과제
1. `build_pi_loop` — 위 그림대로 (채점: 기준값 + blocksim 전 구간 대조)
2. `build_pendulum` — 비선형 진자 + PD (ex2). θ 가 0.8 에 못 가는 이유를
   한 문단으로 쓰라 (가산점: I 를 추가해 오차를 없애 보라)

**다음 주 예고**: 이제 정식 플랫폼으로 갈아탄다 — 8주차, 6-DoF 쿼드로터가
팔레트에서 기다린다. 학기 초의 그 그림이 학기 말에 진짜 기체를 난다.

In [ ]:
# ✅ 채점 — 전부 초록색이 되면 이번 주 완료!
!python -m pytest test_week07.py -q